# Stage 1.2 LSOA Scene Attractiveness

Build the frozen `lsoa_scene_attractiveness.parquet` artifact from cached OSM POIs and UK LSOA boundaries.


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'Modelling').exists() and (candidate / 'Data').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root containing Modelling/ and Data/')


REPO_ROOT = find_repo_root()
MODELLING_DIR = REPO_ROOT / 'Modelling'
NOTEBOOK_DIR = REPO_ROOT / 'Data' / 'Charging_stations' / 'OSM_POI_Labeling'
DATA_DIR = REPO_ROOT / 'Data' / 'Charging_stations'
if str(MODELLING_DIR) not in sys.path:
    sys.path.insert(0, str(MODELLING_DIR))

from mobility.core.constants import SCENE_CATEGORIES
from mobility.cars.build_destination_choice_table import build_lsoa_scene_attractiveness_table

TARGET_CRS = 'EPSG:27700'
POINT_DEFAULT_AREA = 50.0
AREA_MIN = 10.0
AREA_MAX = 50_000.0

OUTPUT_PATH = NOTEBOOK_DIR / 'lsoa_scene_attractiveness.parquet'
CACHE_DIR = NOTEBOOK_DIR / 'cache'
ENGLAND_LSOA = DATA_DIR / 'Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BFE_V10_5330842453253215289.geojson'
SCOTLAND_LSOA = DATA_DIR / 'SG_DataZoneBdry_2022' / 'SG_DataZone_Bdry_2022.shp'
NI_LSOA = DATA_DIR / 'geography-dz2021-geojson' / 'DZ2021.geojson'


In [ ]:
SCENE_TAGS = {
    'home': [
        ('landuse', 'residential', 3),
        ('building', 'residential', 2),
        ('building', 'apartments', 2),
        ('building', 'house', 2),
        ('building', 'detached', 1),
        ('building', 'terrace', 1),
    ],
    'work': [
        ('landuse', 'commercial', 3),
        ('landuse', 'industrial', 2),
        ('building', 'office', 3),
        ('building', 'commercial', 2),
        ('building', 'industrial', 2),
        ('office', True, 2),
        ('amenity', 'workplace', 2),
    ],
    'education': [
        ('amenity', 'school', 3),
        ('amenity', 'university', 3),
        ('amenity', 'college', 3),
        ('amenity', 'kindergarten', 2),
        ('landuse', 'education', 3),
        ('building', 'school', 2),
        ('building', 'university', 2),
    ],
    'shopping': [
        ('shop', True, 3),
        ('landuse', 'retail', 3),
        ('building', 'retail', 2),
        ('building', 'supermarket', 2),
        ('amenity', 'marketplace', 2),
        ('amenity', 'food_court', 1),
    ],
    'personal_business': [
        ('amenity', 'hospital', 3),
        ('amenity', 'clinic', 3),
        ('amenity', 'doctors', 3),
        ('amenity', 'dentist', 2),
        ('amenity', 'pharmacy', 2),
        ('amenity', 'bank', 2),
        ('amenity', 'post_office', 1),
        ('amenity', 'restaurant', 1),
        ('amenity', 'cafe', 1),
        ('amenity', 'fast_food', 1),
        ('amenity', 'bar', 1),
    ],
    'social': [
        ('amenity', 'pub', 2),
        ('amenity', 'bar', 2),
        ('amenity', 'nightclub', 2),
        ('amenity', 'community_centre', 2),
        ('amenity', 'social_facility', 2),
        ('amenity', 'place_of_worship', 1),
        ('building', 'community_centre', 1),
    ],
    'leisure': [
        ('leisure', True, 3),
        ('amenity', 'cinema', 2),
        ('amenity', 'theatre', 2),
        ('amenity', 'sports_centre', 2),
        ('amenity', 'swimming_pool', 2),
        ('sport', True, 2),
        ('landuse', 'recreation_ground', 2),
        ('landuse', 'grass', 1),
        ('natural', 'park', 1),
        ('tourism', 'attraction', 1),
    ],
    'holiday': [
        ('tourism', 'hotel', 3),
        ('tourism', 'motel', 3),
        ('tourism', 'guest_house', 3),
        ('tourism', 'hostel', 2),
        ('tourism', 'camp_site', 2),
        ('tourism', 'caravan_site', 2),
        ('tourism', 'attraction', 1),
        ('tourism', 'museum', 1),
    ],
}


def build_query_tags(scene_tags: dict[str, list[tuple[str, object, int]]]) -> dict[str, object]:
    query_tags: dict[str, object] = {}
    for tag_list in scene_tags.values():
        for key, value, _weight in tag_list:
            existing = query_tags.get(key)
            if value is True:
                query_tags[key] = True
                continue
            if existing is True:
                continue
            if existing is None:
                query_tags[key] = {value}
            else:
                existing.add(value)
    final_tags: dict[str, object] = {}
    for key, value in query_tags.items():
        if value is True:
            final_tags[key] = True
        else:
            sorted_values = sorted(value)
            final_tags[key] = sorted_values[0] if len(sorted_values) == 1 else sorted_values
    return final_tags


def find_poi_cache(cache_dir: Path) -> Path | None:
    candidates = sorted(cache_dir.glob('uk_station_area_pois_*.parquet'))
    if not candidates:
        return None
    return min(candidates, key=lambda path: (path.name.count('-'), len(path.name), path.name))


In [ ]:
poi_cache_path = find_poi_cache(CACHE_DIR)
if poi_cache_path is not None:
    print(f'Loading cached POIs from {poi_cache_path.name}')
    pois = gpd.read_parquet(poi_cache_path)
else:
    from osm_poi_cache import load_station_area_pois

    stations = pd.read_csv(DATA_DIR / 'UK_OCM_stations.csv')
    stations = stations.dropna(subset=['Latitude', 'Longitude']).reset_index(drop=True)
    pois = load_station_area_pois(
        df=stations,
        query_tags=build_query_tags(SCENE_TAGS),
        radius_m=500,
        cache_dir=CACHE_DIR,
        tile_size_m=25_000,
        overpass_timeout=180,
    )

print(f'Total cached POIs: {len(pois):,}')
pois.head(3)


In [ ]:
pois_proj = pois.to_crs(TARGET_CRS).copy()
poi_points = pois_proj.copy()
poi_points = poi_points.set_geometry(pois_proj.geometry.centroid)

is_polygon = pois_proj.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
poi_points['area_m2'] = np.where(is_polygon, pois_proj.geometry.area, POINT_DEFAULT_AREA)
poi_points['area_m2'] = np.clip(poi_points['area_m2'], AREA_MIN, AREA_MAX)

best_scene = np.full(len(poi_points), None, dtype=object)
best_weight = np.full(len(poi_points), -1.0, dtype=float)

for scene, tag_list in SCENE_TAGS.items():
    for key, value, weight in tag_list:
        if key not in poi_points.columns:
            continue
        column = poi_points[key]
        match = column.notna().to_numpy() if value is True else (column == value).to_numpy()
        improve = match & (weight > best_weight)
        best_scene[improve] = scene
        best_weight[improve] = weight

poi_points['scene_label'] = pd.Series(best_scene, index=poi_points.index, dtype='string')
poi_points = poi_points.loc[poi_points['scene_label'].isin(SCENE_CATEGORIES), ['scene_label', 'area_m2', 'geometry']].copy()
print(f'POIs assigned to Stage-1 scenes: {len(poi_points):,}')
poi_points.head(3)


In [ ]:
england_lsoa = gpd.read_file(ENGLAND_LSOA).to_crs(TARGET_CRS)
scotland_lsoa = gpd.read_file(SCOTLAND_LSOA).to_crs(TARGET_CRS)
ni_lsoa = gpd.read_file(NI_LSOA).to_crs(TARGET_CRS)

england_lsoa = england_lsoa.assign(lsoa_code=england_lsoa['LSOA21CD'].astype(str))[['lsoa_code', 'geometry']]
scotland_lsoa = scotland_lsoa.assign(lsoa_code=scotland_lsoa['dzcode'].astype(str))[['lsoa_code', 'geometry']]
ni_lsoa = ni_lsoa.assign(lsoa_code=ni_lsoa['DZ2021_cd'].astype(str))[['lsoa_code', 'geometry']]

uk_lsoa = gpd.GeoDataFrame(
    pd.concat([england_lsoa, scotland_lsoa, ni_lsoa], ignore_index=True),
    geometry='geometry',
    crs=TARGET_CRS,
)
invalid = ~uk_lsoa.is_valid
uk_lsoa.loc[invalid, 'geometry'] = uk_lsoa.loc[invalid, 'geometry'].buffer(0)

poi_with_lsoa = gpd.sjoin(
    poi_points,
    uk_lsoa[['lsoa_code', 'geometry']],
    how='left',
    predicate='within',
).drop(columns=['index_right'])
poi_with_lsoa = poi_with_lsoa[~poi_with_lsoa.index.duplicated(keep='first')].copy()

missing_index = poi_with_lsoa.index[poi_with_lsoa['lsoa_code'].isna()]
print(f'POIs unmatched before nearest fix: {len(missing_index):,}')
if len(missing_index):
    nearest = gpd.sjoin_nearest(
        poi_points.loc[missing_index],
        uk_lsoa[['lsoa_code', 'geometry']],
        how='left',
    ).drop(columns=['index_right'])
    nearest = nearest[~nearest.index.duplicated(keep='first')]
    poi_with_lsoa.loc[missing_index, 'lsoa_code'] = nearest.loc[missing_index, 'lsoa_code']

poi_lsoa = pd.DataFrame(
    {
        'lsoa_code': poi_with_lsoa['lsoa_code'].astype('string').str.strip(),
        'scene_label': poi_with_lsoa['scene_label'].astype('string').str.strip(),
        'area_m2': pd.to_numeric(poi_with_lsoa['area_m2'], errors='coerce'),
    }
)
poi_lsoa = poi_lsoa.dropna(subset=['lsoa_code', 'scene_label', 'area_m2']).reset_index(drop=True)
print(f'POIs available for LSOA aggregation: {len(poi_lsoa):,}')
poi_lsoa.head(3)


In [ ]:
attractiveness_df = build_lsoa_scene_attractiveness_table(poi_lsoa)
attractiveness_df.to_parquet(OUTPUT_PATH, engine='pyarrow', index=False)

print(f'Saved {OUTPUT_PATH.name} with {len(attractiveness_df):,} rows')
print(attractiveness_df.head())
print(attractiveness_df.dtypes)
